## 환경 설정 

In [1]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold

from transformers import AutoProcessor, AutoModelForVision2Seq, T5ForConditionalGeneration, T5Tokenizer

from trl import SFTTrainer, SFTConfig

from datasets import load_dataset, Dataset, load_from_disk, ClassLabel
from peft import LoraConfig, get_peft_model, PeftModel

import gc

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 환경 설정
warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)

✅ Using device: cuda


## 시드고정

In [3]:
# 시드 고정
def seed_everything(seed=47):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

## 모델 선언

In [4]:
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large", device_map='auto', _attn_implementation="eager",)
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large", add_eos_token=True)

# 모델 정보
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"모델 정보:")
print(f"총 파라미터 수: {total_params:,}")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


모델 정보:
총 파라미터 수: 783,150,080


In [5]:
print(model)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 1024)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1024, out_features=2816, bias=False)
              (wi_1): Linear(in_features=1024, out_features=2816, bias=False)
       

In [ ]:
tokenizer.special_tokens_map_extended

{'eos_token': AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 'unk_token': AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 'pad_token': AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 'additional_special_tokens': ['<extra_id_0>',
  '<extra_id_1>',
  '<extra_id_2>',
  '<extra_id_3>',
  '<extra_id_4>',
  '<extra_id_5>',
  '<extra_id_6>',
  '<extra_id_7>',
  '<extra_id_8>',
  '<extra_id_9>',
  '<extra_id_10>',
  '<extra_id_11>',
  '<extra_id_12>',
  '<extra_id_13>',
  '<extra_id_14>',
  '<extra_id_15>',
  '<extra_id_16>',
  '<extra_id_17>',
  '<extra_id_18>',
  '<extra_id_19>',
  '<extra_id_20>',
  '<extra_id_21>',
  '<extra_id_22>',
  '<extra_id_23>',
  '<extra_id_24>',
  '<extra_id_25>',
  '<extra_id_26>',
  '<extra_id_27>',
  '<extra_id_28>',
  '<extra_id_29>',
  '<extra_id_30>',
  '<extra_id_31>',
  '<extra_id_32>',
  '<extra_id_

In [ ]:
# Explanations for CommonsenseQA
class ECQADataset(torch.utils.data.Dataset):
    """
    Explanations for CommonsenseQA
    https://github.com/IBM/ecqa
    """

    def __init__(self, dataset):
        ds = load_dataset("json", data_files="/mnt/workspace/datasets/ecqa/ecqa.jsonl", split="train")
        ds_mapping = {example['id']: example['explanation'] for example in ds}
        def add_explanation(example):
            current_id = example['id']
            example['explanation'] = ds_mapping.get(current_id, None)
            return example
        
        self.dataset = dataset.remove_columns(["question_concept"])
        self.dataset = self.dataset.map(add_explanation)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]

        return {"image": None, 
                "question": row['question'], 
                "choices": row['choices']['text'],
                "answer": row['answerKey'],
                "explanation": row['explanation'],}

In [43]:
class T5DataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, examples):
        '''
        Dataset에서 
        choices는 List[str] 형태이길 기대
        description은 하나의 긴 str이길 기대
        answer은 A, B, C, D 중 하나일 것으로 기대        
        '''
        instructions = []
        answers = []

        for example in examples:
            if not example.get('choices'):  # for RealWorldQA
                instructions.append(f"Read the context and answer the question by choosing the right option among A, B, C, and D.\n Context: {example['description']} \n Question: {example['question']} ")
                answers.append(f"The answer is: ({example['answer']})")
            elif not example.get('explanation'):    # standard QA cases
                formatted_choice = '\n'.join( [f'{chr(65+i)}. {c}' for i, c in enumerate(example['choices']) ] ) 
                instructions.append(f"Read the context and answer the question by choosing the right option among A, B, C, and D.\n Context: {example['description']} \n Question: {example['question']} \n Choices {formatted_choice}")
                answers.append(f"The answer is: ({example['answer']})")
            else:   # for ECQA
                formatted_choice = '\n'.join( [f'{chr(65+i)}. {c}' for i, c in enumerate(example['choices']) ] ) 
                instructions.append(f"Answer the question by choosing the right option among A, B, C, and D by reasoning.\n Question: {example['question']}\n Choices: {formatted_choice}")
                answers.append(f"{example['explanation']} \nThe answer is: ({example['answer']})")

        batch = self.tokenizer(instructions, padding=True, return_tensors="pt")
        labels = self.tokenizer(text_target=answers, padding=True, return_tensors="pt", add_special_tokens=True)
        labels = labels['input_ids']
        
        # 안전성 체크
        if labels is None or len(labels) == 0:
            print(f"ERROR: DataCollator: labels is None or empty. Example inputs that led to this: {examples}")
            print(f"Sample answers: {answers[:3]}")
            raise ValueError("DataCollator produced None or empty labels. Check your data or tokenizer.")
            
        # 패딩 토큰 마스킹: 패딩된 부분은 Loss 계산에서 제외합니다.
        labels[labels == self.tokenizer.pad_token_id] = -100

        # # DEBUG
        # for i in range(len(examples)):
        #     print(f"\n=== DataCollator Debug Info ===")
        #     print(f"3 Sample labels : {labels[:3]}")

        batch["labels"] = labels
        
        return batch

data_collator = T5DataCollator(tokenizer)

In [57]:
ds = load_dataset("tau/commonsense_qa")
train_ds, val_ds = ds['train'], ds['validation']
train_ds, val_ds = ECQADataset(train_ds), ECQADataset(val_ds)
train_ds[3]

Map: 100%|██████████| 1221/1221 [00:00<00:00, 27783.93 examples/s]


{'image': None,
 'question': 'Google Maps and other highway and street GPS services have replaced what?',
 'choices': ['united states', 'mexico', 'countryside', 'atlas', 'oceans'],
 'answer': 'D',
 'explanation': 'Due to Google maps, other highway and street GPS services Atlas has been replaced since they provide better and more accurate data. Other options are places or locations which cannot be replaced by a service like GPS.'}

## SFT Trainer

In [53]:
training_args = SFTConfig(
    report_to='none',
    run_name = 't5-test',
    output_dir="foobar",
    num_train_epochs=1,
    per_device_train_batch_size=3,
    per_device_eval_batch_size=3,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy="epoch",
    optim="adamw_torch_fused",
    bf16=True,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
    eval_do_concat_batches=False,
    label_names=["labels"],
)

In [56]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules= 'all-linear',
    lora_dropout=0.05,
    bias="none",
)

In [58]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_config,
    data_collator=data_collator,
)
trainer.args.predict_with_generate = True
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
output = trainer.predict(val_ds)

## 더미입력으로 확인

In [ ]:
# 더미입력으로 확인
try:
    from PIL import Image
    import numpy as np
    import requests
    
    # PIL 이미지로 더미 이미지 생성 (RGB, 224x224)
    # dummy_image_array = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
    # dummy_image = Image.fromarray(dummy_image_array, mode='RGB')
    url = "https://raw.githubusercontent.com/salesforce/LAVIS/main/docs/_static/Confusing-Pictures.jpg"
    dummy_image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
    dummy_text = "Describe this image."
    
    print("더미 이미지 생성 완료")
    print(f"이미지 크기: {dummy_image.size}, 모드: {dummy_image.mode}")
    
    # 입력 전처리
    inputs = processor(
        images=dummy_image, 
        text=dummy_text, 
        return_tensors="pt"
    )
    
    print("입력 전처리 완료")
    print(f"입력 텐서 키: {list(inputs.keys())}")
    print(f"이미지 텐서 shape: {inputs['pixel_values'].shape}")
    print(f"텍스트 토큰 shape: {inputs['input_ids'].shape}")
    
    # T5 모델을 위한 decoder_input_ids 생성
    # T5는 decoder 시작 시 pad_token_id를 사용
    batch_size = inputs['input_ids'].shape[0]
    decoder_input_ids = torch.full(
        (batch_size, 1), 
        processor.tokenizer.pad_token_id, 
        dtype=torch.long
    )
    inputs['decoder_input_ids'] = decoder_input_ids
    
    print(f"decoder_input_ids 추가: {decoder_input_ids.shape}")
    
    # Forward pass 테스트 (training mode)
    model.train()  # training mode로 설정
    with torch.no_grad():
        outputs = model(**inputs)
    
    print("✓ Forward pass 성공")
    print(f"출력 logits shape: {outputs.logits.shape}")
    print(f"출력 logits 범위: [{outputs.logits.min().item():.4f}, {outputs.logits.max().item():.4f}]")
    
    # Generation 테스트도 수행
    print("\nGeneration 테스트...")
    model.eval()  # evaluation mode로 설정
    with torch.no_grad():
        # decoder_input_ids 제거 (generate에서는 자동 생성)
        gen_inputs = {k: v for k, v in inputs.items() if k != 'decoder_input_ids'}
        generated_ids = model.generate(
            **gen_inputs,
            max_new_tokens=30,
            do_sample=True,
            num_beams=3,
            repetition_penalty=1.5,
        )
    
    # 생성된 텍스트 디코딩
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(f"✓ Generation 성공")
    print(f"생성된 텍스트 (샘플): '{generated_text}...'")
      
    
except Exception as e:
    print(f"✗ 테스트 실패: {str(e)}")
    # 에러 발생 시 더 자세한 디버깅 정보 제공
    import traceback
    print("상세 에러 정보:")
    traceback.print_exc()


print("\n모델 개조 및 테스트 완료!")

## Inference sample

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq


# model = AutoModelForVision2Seq.from_pretrained("microsoft/kosmos-2-patch14-224")
# processor = AutoProcessor.from_pretrained("microsoft/kosmos-2-patch14-224")
# # --------------------- [Optional] 모델 로드 for continual training -----------------------
final_save_directory = os.path.join("/mnt/workspace/Model/kosmos2/2nd_phase/final_best_model")
model = Kosmos2ForConditionalGeneration.from_pretrained("microsoft/kosmos-2-patch14-224", torch_dtype=torch.bfloat16, device_map="auto" )   # 베이스모델 로드
model = PeftModel.from_pretrained(model, final_save_directory, torch_dtype=torch.bfloat16, device_map="auto", is_trainable=True )       # LoRA weight 적용된 모델 로드
processor = AutoProcessor.from_pretrained(final_save_directory, add_eos_token=True)

In [ ]:
from IPython.display import clear_output

# ds = load_dataset("json", data_files="../../eg/train_aug.jsonl", split='train')
ds = pd.read_csv('../../eg/test.csv')[0:10]

for i ,row in ds.iterrows():
    path = os.path.join('/mnt/workspace/eg', row['img_path'])
    image = Image.open(path)
    
    inputs = processor(text="Describe this image in detail.", images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values=inputs["pixel_values"].to(device),
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            image_embeds=None,
            image_embeds_position_mask=inputs["image_embeds_position_mask"],
            use_cache=True,
            max_new_tokens=2048,
            do_sample=False,
            num_beams=3
        )
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    processed_text, entities = processor.post_process_generation(generated_text)
    print(processed_text,'\n')
    image.show()
    # print(row['description'])
    input(f"[{i}/{len(ds)}] 다음을 보려면 Enter를 누르세요...")
    clear_output(wait=True)